In [2]:
#!/usr/bin/env python3
"""
Interactive feature-by-feature validation of noun/proper-noun semantic tags.

Compatible with Python 3.9.21.

Input:
    ewt_wordnet_wsd_features.jsonl

Behaviour:
    - Randomly selects 5% of UNIQUE sentences containing at least one NOUN/PROPN.
    - Saves the sample so subsequent runs use exactly the same sentences.
    - Displays each complete sampled sentence.
    - Displays every NOUN and PROPN in that sentence.
    - Reviews EACH assigned feature individually:
          c = correct
          i = incorrect
          s = unsure / skip
          q = save and quit
    - Tokens with no features are automatically recorded as incorrect.
    - Saves every feature judgement immediately.
    - Can safely resume after interruption.

No external packages are required.
"""

from __future__ import annotations

import csv
import json
import math
import random
from collections import defaultdict
from pathlib import Path
from typing import Any, Dict, List, Set, Tuple


# =============================================================================
# Configuration
# =============================================================================

INPUT_PATH = Path("ewt_wordnet_wsd_features_Qwen35b_a3b_mtp_2026_08_15.jsonl")

SAMPLE_FRACTION = 0.05
RANDOM_SEED = 20260818

SAMPLE_PATH = Path("ewt_noun_validation_sample.json")
RESULTS_PATH = Path("ewt_noun_feature_validation_results.csv")

TARGET_POS = {"NOUN", "PROPN"}

# Used to represent a token for which the model assigned nothing.
NO_FEATURES_MARKER = "__NO_FEATURES__"


# =============================================================================
# Input loading
# =============================================================================

def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    """Load valid JSON objects from a JSONL file."""

    if not path.exists():
        raise FileNotFoundError(
            f"Could not find input file:\n{path.resolve()}"
        )

    records: List[Dict[str, Any]] = []

    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                print(
                    f"Warning: ignoring invalid JSON on line "
                    f"{line_number}: {exc}"
                )
                continue

            if isinstance(record, dict):
                records.append(record)

    return records


def token_sort_key(value: Any) -> Tuple[int, str]:
    """Sort normal integer token IDs before unusual IDs."""

    if isinstance(value, int):
        return value, ""

    return 10**9, str(value)


def group_target_records(
    records: List[Dict[str, Any]],
) -> Dict[int, List[Dict[str, Any]]]:
    """Group NOUN/PROPN records by sentence index."""

    grouped: Dict[int, List[Dict[str, Any]]] = defaultdict(list)

    for record in records:
        if record.get("upos") not in TARGET_POS:
            continue

        sentence_index = record.get("sentence_index")

        if not isinstance(sentence_index, int):
            continue

        grouped[sentence_index].append(record)

    for sentence_records in grouped.values():
        sentence_records.sort(
            key=lambda r: token_sort_key(r.get("token_id"))
        )

    return dict(grouped)


# =============================================================================
# Sampling
# =============================================================================

def create_or_load_sample(
    grouped: Dict[int, List[Dict[str, Any]]],
) -> List[int]:
    """
    Create a random sample of 5% of eligible sentences.

    Once created, the exact same sample is reused on subsequent runs.
    """

    eligible_indices = sorted(grouped)

    if not eligible_indices:
        raise RuntimeError(
            "No sentences containing NOUN or PROPN records were found."
        )

    if SAMPLE_PATH.exists():
        with SAMPLE_PATH.open("r", encoding="utf-8") as handle:
            saved = json.load(handle)

        sample_indices = [
            index
            for index in saved.get("sentence_indices", [])
            if isinstance(index, int) and index in grouped
        ]

        if not sample_indices:
            raise RuntimeError(
                f"{SAMPLE_PATH} exists but contains no usable "
                "sentence indices."
            )

        print(
            f"Loaded existing validation sample: "
            f"{len(sample_indices):,} sentences."
        )

        return sample_indices

    sample_size = max(
        1,
        math.ceil(len(eligible_indices) * SAMPLE_FRACTION),
    )

    rng = random.Random(RANDOM_SEED)

    sample_indices = rng.sample(
        eligible_indices,
        k=min(sample_size, len(eligible_indices)),
    )

    sample_data = {
        "sample_fraction": SAMPLE_FRACTION,
        "random_seed": RANDOM_SEED,
        "eligible_sentence_count": len(eligible_indices),
        "sample_sentence_count": len(sample_indices),
        "sentence_indices": sample_indices,
    }

    with SAMPLE_PATH.open("w", encoding="utf-8") as handle:
        json.dump(
            sample_data,
            handle,
            ensure_ascii=False,
            indent=2,
        )

    print(
        f"Created new validation sample: "
        f"{len(sample_indices):,} of "
        f"{len(eligible_indices):,} eligible sentences."
    )

    return sample_indices


# =============================================================================
# Feature helpers
# =============================================================================

def get_features(record: Dict[str, Any]) -> List[str]:
    """Return a clean, stable feature list."""

    features = record.get("features", [])

    if not isinstance(features, list):
        return []

    cleaned: List[str] = []

    for feature in features:
        feature_text = str(feature).strip()

        if feature_text and feature_text not in cleaned:
            cleaned.append(feature_text)

    return cleaned


def feature_key(
    sentence_index: int,
    token_id: Any,
    feature: str,
) -> Tuple[int, str, str]:
    """Stable key identifying one token-feature judgement."""

    return (
        sentence_index,
        str(token_id),
        feature,
    )


# =============================================================================
# Existing results
# =============================================================================

def load_completed_judgements(
    path: Path,
) -> Set[Tuple[int, str, str]]:
    """Recover completed feature judgements."""

    completed: Set[Tuple[int, str, str]] = set()

    if not path.exists():
        return completed

    with path.open(
        "r",
        encoding="utf-8",
        newline="",
    ) as handle:
        reader = csv.DictReader(handle)

        for row in reader:
            try:
                sentence_index = int(row["sentence_index"])
            except (KeyError, TypeError, ValueError):
                continue

            token_id = row.get("token_id")
            feature = row.get("feature")

            if token_id is None or feature is None:
                continue

            completed.add(
                feature_key(
                    sentence_index,
                    token_id,
                    feature,
                )
            )

    return completed


# =============================================================================
# Result writing
# =============================================================================

FIELDNAMES = [
    "sentence_index",
    "sentence_id",
    "sentence_text",
    "token_id",
    "form",
    "lemma",
    "upos",
    "selected_synset",
    "selected_definition",
    "feature",
    "judgement",
    "automatic",
]


def append_judgement(
    record: Dict[str, Any],
    feature: str,
    judgement: str,
    automatic: bool = False,
) -> None:
    """Append one feature-level judgement."""

    file_exists = RESULTS_PATH.exists()

    with RESULTS_PATH.open(
        "a",
        encoding="utf-8",
        newline="",
    ) as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=FIELDNAMES,
        )

        if not file_exists:
            writer.writeheader()

        writer.writerow(
            {
                "sentence_index": record.get("sentence_index"),
                "sentence_id": record.get("sentence_id"),
                "sentence_text": record.get("sentence_text"),
                "token_id": record.get("token_id"),
                "form": record.get("form"),
                "lemma": record.get("lemma"),
                "upos": record.get("upos"),
                "selected_synset": record.get("selected_synset"),
                "selected_definition": record.get(
                    "selected_definition"
                ),
                "feature": feature,
                "judgement": judgement,
                "automatic": automatic,
            }
        )


# =============================================================================
# Display
# =============================================================================

def separator(character: str = "=", width: int = 88) -> None:
    print(character * width)


def display_sentence(
    sample_number: int,
    total_sentences: int,
    sentence_index: int,
    records: List[Dict[str, Any]],
) -> None:
    """Show the sentence and all noun/proper-noun classifications."""

    separator()

    sentence_text = ""

    for record in records:
        if record.get("sentence_text"):
            sentence_text = str(record["sentence_text"])
            break

    sentence_id = (
        records[0].get("sentence_id")
        if records
        else None
    )

    print(
        f"SAMPLE SENTENCE "
        f"{sample_number}/{total_sentences}"
    )
    print(f"Corpus sentence index: {sentence_index}")

    if sentence_id:
        print(f"Sentence ID: {sentence_id}")

    print()
    print(sentence_text)
    print()

    print("NOUNS / PROPER NOUNS:")
    print()

    for number, record in enumerate(records, start=1):
        features = get_features(record)

        feature_text = (
            ", ".join(features)
            if features
            else "[NO FEATURES]"
        )

        print(
            f"  {number}. "
            f"{record.get('form')!r} "
            f"({record.get('upos')})"
        )

        print(f"     {feature_text}")

    print()


def display_token(
    record: Dict[str, Any],
) -> None:
    """Display detailed information for one token."""

    separator("-")

    print(
        f"{record.get('form')!r} "
        f"[{record.get('upos')}]"
    )

    print(f"Position: {record.get('token_id')}")

    if record.get("lemma"):
        print(f"Lemma: {record.get('lemma')}")

    selected_synset = record.get("selected_synset")
    selected_definition = record.get("selected_definition")

    if selected_synset:
        print()
        print(f"WordNet sense: {selected_synset}")

        if selected_definition:
            print(
                f"Definition: {selected_definition}"
            )

    print()


# =============================================================================
# Interactive input
# =============================================================================

def ask_feature_judgement(
    feature: str,
) -> str:
    """Ask whether one assigned feature is correct."""

    while True:
        answer = input(
            f"{feature}: "
            "[c]orrect / [i]ncorrect / "
            "[s]unsure / [q]uit: "
        ).strip().lower()

        mapping = {
            "c": "correct",
            "correct": "correct",

            "i": "incorrect",
            "incorrect": "incorrect",

            "s": "unsure",
            "skip": "unsure",
            "unsure": "unsure",

            "q": "quit",
            "quit": "quit",
        }

        if answer in mapping:
            return mapping[answer]

        print("Please enter c, i, s, or q.")


# =============================================================================
# Work counting
# =============================================================================

def expected_feature_keys(
    sample_indices: List[int],
    grouped: Dict[int, List[Dict[str, Any]]],
) -> Set[Tuple[int, str, str]]:
    """
    Return all judgements expected from the sampled data.

    A token with no features contributes one __NO_FEATURES__ item.
    """

    keys: Set[Tuple[int, str, str]] = set()

    for sentence_index in sample_indices:
        for record in grouped.get(sentence_index, []):
            token_id = record.get("token_id")
            features = get_features(record)

            if features:
                for feature in features:
                    keys.add(
                        feature_key(
                            sentence_index,
                            token_id,
                            feature,
                        )
                    )
            else:
                keys.add(
                    feature_key(
                        sentence_index,
                        token_id,
                        NO_FEATURES_MARKER,
                    )
                )

    return keys


# =============================================================================
# Review
# =============================================================================

def review_sample(
    sample_indices: List[int],
    grouped: Dict[int, List[Dict[str, Any]]],
) -> None:
    """Run the feature-by-feature validation session."""

    completed = load_completed_judgements(
        RESULTS_PATH
    )

    expected = expected_feature_keys(
        sample_indices,
        grouped,
    )

    already_done = len(
        completed.intersection(expected)
    )

    separator()
    print("NOUN / PROPER-NOUN FEATURE VALIDATION")
    separator()

    print(f"Sample sentences: {len(sample_indices):,}")
    print(f"Feature judgements required: {len(expected):,}")
    print(f"Already completed: {already_done:,}")
    print(
        f"Remaining: "
        f"{len(expected) - already_done:,}"
    )
    print()

    reviewed_this_run = 0
    automatically_wrong_this_run = 0

    for sample_number, sentence_index in enumerate(
        sample_indices,
        start=1,
    ):
        records = grouped.get(sentence_index, [])

        # Determine whether anything in the sentence remains.
        sentence_has_pending = False

        for record in records:
            features = get_features(record)

            if not features:
                features = [NO_FEATURES_MARKER]

            for feature in features:
                key = feature_key(
                    sentence_index,
                    record.get("token_id"),
                    feature,
                )

                if key not in completed:
                    sentence_has_pending = True
                    break

            if sentence_has_pending:
                break

        if not sentence_has_pending:
            continue

        display_sentence(
            sample_number=sample_number,
            total_sentences=len(sample_indices),
            sentence_index=sentence_index,
            records=records,
        )

        for record in records:
            features = get_features(record)

            # -------------------------------------------------------------
            # No features: automatically wrong
            # -------------------------------------------------------------

            if not features:
                key = feature_key(
                    sentence_index,
                    record.get("token_id"),
                    NO_FEATURES_MARKER,
                )

                if key in completed:
                    continue

                display_token(record)

                print(
                    "NO FEATURES ASSIGNED "
                    "→ automatically marked INCORRECT."
                )
                print()

                append_judgement(
                    record=record,
                    feature=NO_FEATURES_MARKER,
                    judgement="incorrect",
                    automatic=True,
                )

                completed.add(key)

                reviewed_this_run += 1
                automatically_wrong_this_run += 1

                continue

            # -------------------------------------------------------------
            # Review every assigned feature separately
            # -------------------------------------------------------------

            pending_features = [
                feature
                for feature in features
                if feature_key(
                    sentence_index,
                    record.get("token_id"),
                    feature,
                ) not in completed
            ]

            if not pending_features:
                continue

            display_token(record)

            print("ASSIGNED FEATURES:")
            print()

            for feature in features:
                key = feature_key(
                    sentence_index,
                    record.get("token_id"),
                    feature,
                )

                state = (
                    " [already reviewed]"
                    if key in completed
                    else ""
                )

                print(
                    f"  - {feature}{state}"
                )

            print()

            for feature in pending_features:
                judgement = ask_feature_judgement(
                    feature
                )

                if judgement == "quit":
                    separator()
                    print(
                        "Validation stopped. "
                        "All previous judgements are saved."
                    )
                    print(
                        f"Results:\n"
                        f"  {RESULTS_PATH.resolve()}"
                    )
                    return

                append_judgement(
                    record=record,
                    feature=feature,
                    judgement=judgement,
                    automatic=False,
                )

                completed.add(
                    feature_key(
                        sentence_index,
                        record.get("token_id"),
                        feature,
                    )
                )

                reviewed_this_run += 1

                print(
                    f"Saved {feature}: "
                    f"{judgement}"
                )

            print()

    separator()
    print("VALIDATION SAMPLE COMPLETE")
    separator()

    print(
        f"Judgements recorded this run: "
        f"{reviewed_this_run:,}"
    )

    print(
        f"Automatically incorrect "
        f"(no features): "
        f"{automatically_wrong_this_run:,}"
    )

    print(
        f"Results written to:\n"
        f"  {RESULTS_PATH.resolve()}"
    )


# =============================================================================
# Main
# =============================================================================

def main() -> None:
    print(
        f"Loading:\n"
        f"  {INPUT_PATH.resolve()}"
    )

    records = load_jsonl(INPUT_PATH)

    print(
        f"Loaded {len(records):,} token records."
    )

    grouped = group_target_records(records)

    noun_prop_count = sum(
        len(sentence_records)
        for sentence_records in grouped.values()
    )

    print(
        f"Found {noun_prop_count:,} NOUN/PROPN records "
        f"across {len(grouped):,} sentences."
    )

    sample_indices = create_or_load_sample(
        grouped
    )

    review_sample(
        sample_indices=sample_indices,
        grouped=grouped,
    )


if __name__ == "__main__":
    main()

Loading:
  /home/mark/AIML339/ewt_wordnet_wsd_features_Qwen35b_a3b_mtp_2026_08_15.jsonl
Loaded 18,209 token records.
Found 13,088 NOUN/PROPN records across 2,757 sentences.
Created new validation sample: 138 of 2,757 eligible sentences.
NOUN / PROPER-NOUN FEATURE VALIDATION
Sample sentences: 138
Feature judgements required: 2,056
Already completed: 0
Remaining: 2,056

SAMPLE SENTENCE 1/138
Corpus sentence index: 1430
Sentence ID: weblog-blogspot.com_dakbangla_20050311135387_ENG_20050311_135387-0081

In the case of the anthrax used in the mailings, there was an electrostatic charge.

NOUNS / PROPER NOUNS:

  1. 'case' (NOUN)
     Abstract
  2. 'anthrax' (NOUN)
     Abstract
  3. 'mailings' (NOUN)
     Abstract
  4. 'charge' (NOUN)
     Concrete, Inanimate, Internal_Control, Natural, Natural_Force

----------------------------------------------------------------------------------------
'case' [NOUN]
Position: 3
Lemma: case

WordNet sense: event.n.02
Definition: a special set of circumsta

Abstract: [c]orrect / [i]ncorrect / [s]unsure / [q]uit:  c


Saved Abstract: correct

----------------------------------------------------------------------------------------
'anthrax' [NOUN]
Position: 6
Lemma: anthrax

WordNet sense: anthrax.n.01
Definition: a highly infectious animal disease (especially cattle and sheep); it can be transmitted to people

ASSIGNED FEATURES:

  - Abstract



Abstract: [c]orrect / [i]ncorrect / [s]unsure / [q]uit:  i


Saved Abstract: incorrect

----------------------------------------------------------------------------------------
'mailings' [NOUN]
Position: 10
Lemma: mailing

WordNet sense: mailing.n.01
Definition: mail sent by a sender at one time

ASSIGNED FEATURES:

  - Abstract



...

Saved Natural: correct


Natural_Force: [c]orrect / [i]ncorrect / [s]unsure / [q]uit:  c


Saved Natural_Force: correct

SAMPLE SENTENCE 2/138
Corpus sentence index: 444
Sentence ID: weblog-blogspot.com_healingiraq_20050121235804_ENG_20050121_235804-0039

I concur that Ba'athists and former security forces are capable of immediately controlling at least 5 out of 18 governorates, along with the capital, if Americans are to be removed from the picture entirely.

NOUNS / PROPER NOUNS:

  1. "Ba'athists" (PROPN)
     Abstract, Internal_Control, Organisation
  2. 'security' (NOUN)
     [NO FEATURES]
  3. 'forces' (NOUN)
     Abstract, Internal_Control, Organisation
  4. 'governorates' (NOUN)
     [NO FEATURES]
  5. 'capital' (NOUN)
     Concrete, Inanimate, Location
  6. 'Americans' (PROPN)
     Animate, Concrete, Human, Natural, Unspecified_Gender_or_Neuter
  7. 'picture' (NOUN)
     Abstract

----------------------------------------------------------------------------------------
"Ba'athists" [PROPN]
Position: 4
Lemma: Ba'athist

ASSIGNED FEATURES:

  - Abstract
  - Internal_Co

Abstract: [c]orrect / [i]ncorrect / [s]unsure / [q]uit:  q


Validation stopped. All previous judgements are saved.
Results:
  /home/mark/AIML339/ewt_noun_feature_validation_results.csv


In [3]:
#!/usr/bin/env python3
"""
Group NOUN and PROPN occurrences by their exact assigned semantic feature sets.

Compatible with Python 3.9.21.

Input:
    ewt_wordnet_wsd_features.jsonl

Output:
    ewt_noun_feature_groups.html

Features:
    - Groups NOUN and PROPN records by exact feature set.
    - Groups records with no features separately and marks them as errors.
    - Within each feature set, groups occurrences by lemma/form.
    - Shows frequency counts.
    - Shows sentence context for each occurrence.
    - Shows WordNet sense/definition for common nouns where available.
    - Sorts large groups first.
    - Makes anomalous assignments easier to spot visually.

No external packages are required.
"""

from __future__ import annotations

import html
import json
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, List, Tuple


# =============================================================================
# Configuration
# =============================================================================

INPUT_PATH = Path("ewt_wordnet_wsd_features_Qwen35b_a3b_mtp_2026_08_15.jsonl")
OUTPUT_PATH = Path("ewt_noun_feature_groups_Qwen35b_a3b_mtp_2026_08_15.html")

TARGET_POS = {"NOUN", "PROPN"}

NO_FEATURES_LABEL = "(NO FEATURES)"


# =============================================================================
# Loading
# =============================================================================

def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    """Load JSONL records."""

    if not path.exists():
        raise FileNotFoundError(
            f"Could not find:\n{path.resolve()}"
        )

    records: List[Dict[str, Any]] = []

    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                print(
                    f"Warning: invalid JSON on line "
                    f"{line_number}: {exc}"
                )
                continue

            if not isinstance(record, dict):
                continue

            if record.get("upos") not in TARGET_POS:
                continue

            records.append(record)

    return records


# =============================================================================
# Feature handling
# =============================================================================

def get_feature_tuple(
    record: Dict[str, Any],
) -> Tuple[str, ...]:
    """
    Return the exact feature set as a sorted tuple.

    Sorting means that:
        ["Human", "Animate"]
    and:
        ["Animate", "Human"]

    belong to the same group.
    """

    features = record.get("features", [])

    if not isinstance(features, list):
        return ()

    cleaned = {
        str(feature).strip()
        for feature in features
        if str(feature).strip()
    }

    return tuple(sorted(cleaned))


def display_feature_set(
    features: Tuple[str, ...],
) -> str:
    if not features:
        return NO_FEATURES_LABEL

    return " | ".join(features)


# =============================================================================
# Word grouping
# =============================================================================

def word_key(record: Dict[str, Any]) -> str:
    """
    Prefer lemma for common nouns and form for proper nouns.

    Proper noun capitalisation can be semantically important, so retain form.
    """

    if record.get("upos") == "PROPN":
        value = record.get("form")
    else:
        value = record.get("lemma") or record.get("form")

    return str(value or "").strip()


def group_records(
    records: List[Dict[str, Any]],
) -> Dict[
    Tuple[str, ...],
    Dict[str, List[Dict[str, Any]]]
]:
    """
    feature-set -> word -> occurrences
    """

    grouped = defaultdict(
        lambda: defaultdict(list)
    )

    for record in records:
        features = get_feature_tuple(record)
        word = word_key(record)

        grouped[features][word].append(record)

    return grouped


# =============================================================================
# HTML helpers
# =============================================================================

def h(value: Any) -> str:
    """HTML escape."""

    return html.escape(str(value or ""))


def make_anchor(
    feature_number: int,
) -> str:
    return f"group-{feature_number}"


# =============================================================================
# HTML report
# =============================================================================

def create_report(
    records: List[Dict[str, Any]],
    grouped: Dict[
        Tuple[str, ...],
        Dict[str, List[Dict[str, Any]]]
    ],
) -> str:

    total_records = len(records)
    total_groups = len(grouped)

    no_feature_count = sum(
        len(occurrences)
        for occurrences in grouped.get((), {}).values()
    )

    # Sort feature groups by number of token occurrences.
    sorted_groups = sorted(
        grouped.items(),
        key=lambda item: (
            -sum(
                len(occurrences)
                for occurrences in item[1].values()
            ),
            display_feature_set(item[0]).lower(),
        ),
    )

    parts: List[str] = []

    parts.append(
        """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Noun Feature Groups</title>

<style>

body {
    font-family: Arial, Helvetica, sans-serif;
    max-width: 1500px;
    margin: 30px auto;
    padding: 0 30px;
    line-height: 1.45;
    color: #222;
}

h1 {
    margin-bottom: 0.3em;
}

h2 {
    margin-top: 2.5em;
    border-bottom: 2px solid #999;
    padding-bottom: 0.25em;
}

h3 {
    margin-bottom: 0.3em;
}

.summary {
    background: #f4f4f4;
    padding: 16px;
    border-radius: 6px;
    margin: 20px 0;
}

.no-features {
    background: #ffe5e5;
    border-left: 6px solid #b00020;
    padding: 12px 18px;
}

.feature-group {
    margin-bottom: 40px;
}

.word {
    margin: 12px 0;
    padding: 10px 14px;
    background: #fafafa;
    border-left: 4px solid #bbb;
}

.word-name {
    font-size: 1.1em;
    font-weight: bold;
}

.count {
    color: #666;
    font-size: 0.9em;
}

.occurrence {
    margin: 8px 0 8px 20px;
    padding: 8px 12px;
    border-left: 2px solid #ddd;
}

.sentence {
    font-family: Georgia, serif;
}

.metadata {
    color: #555;
    font-size: 0.9em;
    margin-top: 3px;
}

.sense {
    font-size: 0.9em;
    margin-top: 4px;
}

details {
    margin-top: 6px;
}

summary {
    cursor: pointer;
    color: #444;
}

table {
    border-collapse: collapse;
    width: 100%;
}

th, td {
    text-align: left;
    border-bottom: 1px solid #ddd;
    padding: 6px 8px;
}

th {
    background: #eee;
}

a {
    color: #184b8a;
}

</style>
</head>
<body>
"""
    )

    parts.append(
        "<h1>Noun / Proper-Noun Feature Groups</h1>"
    )

    parts.append(
        f"""
<div class="summary">
<strong>Total noun/proper-noun records:</strong>
{total_records:,}<br>

<strong>Exact feature sets:</strong>
{total_groups:,}<br>

<strong>Records with no features:</strong>
{no_feature_count:,}
</div>
"""
    )

    # -------------------------------------------------------------------------
    # Table of contents
    # -------------------------------------------------------------------------

    parts.append("<h2>Feature-set index</h2>")

    parts.append(
        """
<table>
<tr>
<th>Feature set</th>
<th>Distinct words</th>
<th>Occurrences</th>
</tr>
"""
    )

    for group_number, (features, words) in enumerate(
        sorted_groups,
        start=1,
    ):
        occurrence_count = sum(
            len(items)
            for items in words.values()
        )

        class_name = (
            ' class="no-features"'
            if not features
            else ""
        )

        parts.append(
            f"""
<tr{class_name}>
<td>
<a href="#{make_anchor(group_number)}">
{h(display_feature_set(features))}
</a>
</td>
<td>{len(words):,}</td>
<td>{occurrence_count:,}</td>
</tr>
"""
        )

    parts.append("</table>")

    # -------------------------------------------------------------------------
    # Feature groups
    # -------------------------------------------------------------------------

    for group_number, (features, words) in enumerate(
        sorted_groups,
        start=1,
    ):
        occurrence_count = sum(
            len(items)
            for items in words.values()
        )

        extra_class = (
            " no-features"
            if not features
            else ""
        )

        parts.append(
            f"""
<div
    class="feature-group{extra_class}"
    id="{make_anchor(group_number)}"
>
<h2>{h(display_feature_set(features))}</h2>

<p>
<strong>{len(words):,}</strong> distinct words;
<strong>{occurrence_count:,}</strong> occurrences.
</p>
"""
        )

        if not features:
            parts.append(
                """
<p>
<strong>
These tokens received no semantic features and are therefore
automatically considered incorrect under the validation criterion.
</strong>
</p>
"""
            )

        # Sort words primarily by frequency, secondarily alphabetically.
        sorted_words = sorted(
            words.items(),
            key=lambda item: (
                -len(item[1]),
                item[0].lower(),
            ),
        )

        for word, occurrences in sorted_words:
            pos_counts = Counter(
                str(record.get("upos") or "")
                for record in occurrences
            )

            pos_text = ", ".join(
                f"{pos}: {count}"
                for pos, count in sorted(pos_counts.items())
            )

            parts.append(
                f"""
<div class="word">

<div class="word-name">
{h(word)}
<span class="count">
({len(occurrences):,} occurrences; {h(pos_text)})
</span>
</div>
"""
            )

            # Collapse contexts so that large groups remain browsable.
            parts.append(
                f"""
<details>
<summary>
Show {len(occurrences):,} occurrence(s)
</summary>
"""
            )

            for record in occurrences:
                sentence = record.get(
                    "sentence_text",
                    ""
                )

                form = record.get(
                    "form",
                    ""
                )

                sentence_index = record.get(
                    "sentence_index",
                    ""
                )

                token_id = record.get(
                    "token_id",
                    ""
                )

                upos = record.get(
                    "upos",
                    ""
                )

                selected_synset = record.get(
                    "selected_synset"
                )

                selected_definition = record.get(
                    "selected_definition"
                )

                parts.append(
                    """
<div class="occurrence">
"""
                )

                parts.append(
                    f"""
<div class="sentence">
{h(sentence)}
</div>
"""
                )

                parts.append(
                    f"""
<div class="metadata">
Token: <strong>{h(form)}</strong>
&nbsp;|&nbsp;
POS: {h(upos)}
&nbsp;|&nbsp;
Sentence: {h(sentence_index)}
&nbsp;|&nbsp;
Token position: {h(token_id)}
</div>
"""
                )

                if selected_synset:
                    parts.append(
                        f"""
<div class="sense">
<strong>{h(selected_synset)}</strong>
"""
                    )

                    if selected_definition:
                        parts.append(
                            f" — {h(selected_definition)}"
                        )

                    parts.append("</div>")

                parts.append("</div>")

            parts.append("</details>")
            parts.append("</div>")

        parts.append("</div>")

    parts.append(
        """
</body>
</html>
"""
    )

    return "".join(parts)


# =============================================================================
# Main
# =============================================================================

def main() -> None:

    print(f"Reading: {INPUT_PATH.resolve()}")

    records = load_jsonl(INPUT_PATH)

    print(
        f"Loaded {len(records):,} "
        "NOUN/PROPN records."
    )

    grouped = group_records(records)

    print(
        f"Found {len(grouped):,} "
        "unique feature sets."
    )

    no_feature_count = sum(
        len(items)
        for items in grouped.get((), {}).values()
    )

    print(
        f"No-feature records: "
        f"{no_feature_count:,}"
    )

    report = create_report(
        records=records,
        grouped=grouped,
    )

    OUTPUT_PATH.write_text(
        report,
        encoding="utf-8",
    )

    print()
    print(
        f"Report written to:\n"
        f"  {OUTPUT_PATH.resolve()}"
    )


if __name__ == "__main__":
    main()

Reading: /home/mark/AIML339/ewt_wordnet_wsd_features_Qwen35b_a3b_mtp_2026_08_15.jsonl
Loaded 13,088 NOUN/PROPN records.
Found 181 unique feature sets.
No-feature records: 1,051

Report written to:
  /home/mark/AIML339/ewt_noun_feature_groups_Qwen35b_a3b_mtp_2026_08_15.html
